# One-Rec — ranker training v11 (audited consolidated run)

Implements the external audit in one session: true end-to-end metrics (all
original positives in the denominator, dropped groups kept), corrected
seed-only construction, full NCF split exclusion, deterministic per-playlist
manifests, serving-exact policy grid tuned on validation, ranker ablation
variants, v9 baseline rescoring, and SHA-256 provenance.

**Inputs:** MPD mirror; `music-rec-base-models` (item2vec + mood +
features_spec); `one-rec-audio-ckpt` (audio_emb.parquet + v9_baseline_ranker.txt);
`one-rec-ann-index` (the exact installed 50-tree Annoy artifact). GPU T4.


In [ ]:
import glob, hashlib, json, os, pickle, sys, time, zipfile
from pathlib import Path

import numpy as np

%pip install -q annoy
import annoy, gensim, lightgbm as lgb, pandas as pd, sklearn, torch
print("numpy", np.__version__, "| gensim", gensim.__version__, "| lightgbm", lgb.__version__,
      "| torch", torch.__version__, "| cuda:", torch.cuda.is_available())


In [ ]:
CFG = dict(
    mpd_glob="/kaggle/input/**/mpd.slice.*.json",
    min_playlist_len=10, max_playlist_len=250, min_artists=3,

    ncf_min_count=15, ncf_gmf_dim=64, ncf_mlp_dim=64, ncf_mlp_layers=[128, 64],
    ncf_epochs=3, ncf_samples_per_epoch=3_000_000, ncf_batch=4096, ncf_lr=1e-3,
    ncf_ctx_min=5, ncf_ctx_max=20,

    ranker_train=25_000, ranker_val=2_500, ranker_test=5_000,
    retrieval_pool=1_000,
    seed_share=0.70,      # funnel ablation: .4714 retrieved-positive recall vs .4273 at .85
    artist_precap=None,   # ablation: pre-caps destroyed recall; service post-rank rule owns diversity
    cand_cap=1_000,       # == uncapped at pool size
    mood_keep=1.0,
    seed_only_frac=0.25,
    labels="binary_playlist_membership",   # graded kept only as a named diagnostic
    seed=42,

    # policy grid (validation-only; frozen before test scoring)
    grid_lambda=[0.0, 0.5, 1.0, 1.5, 2.2, 3.0],
    grid_mu=[0.0, 0.5, 1.0, 1.5, 2.0],
    grid_floor=[0.0, 0.20, 0.25, 0.30],
    policy_seed_cos_min=0.68,   # v9 served seed_cos@10 constraint (vibe non-regression)
    serve_limit=10, serve_overfetch=3,
)

WORK = Path("/kaggle/working")
hits = glob.glob("/kaggle/input/**/item2vec.wordvectors", recursive=True)
assert hits, "attach the music-rec-base-models dataset"
BASE = Path(hits[0]).parent
sys.path.insert(0, str(BASE))
import features_spec as F
assert len(F.FEATURE_NAMES) == 20, f"stale features_spec ({len(F.FEATURE_NAMES)} features)"
print("feature contract:", len(F.FEATURE_NAMES), "features")


## Stage 1 — parse MPD → playlists + `track_meta.parquet`

In [ ]:
meta_path, playlists_path = WORK / "track_meta.parquet", WORK / "playlists.npz"

if meta_path.exists() and playlists_path.exists():
    print("stage 1 cached")
else:
    slices = sorted(glob.glob(CFG["mpd_glob"], recursive=True))
    assert slices, "no mpd.slice.*.json found — attach an MPD mirror dataset"
    print(f"{len(slices)} slices")
    meta, id_of, playlists = {}, {}, []
    t0 = time.time()
    for si, path in enumerate(slices):
        with open(path) as f:
            data = json.load(f)
        for pl in data["playlists"]:
            tracks = pl.get("tracks", [])
            if not (CFG["min_playlist_len"] <= len(tracks) <= CFG["max_playlist_len"]):
                continue
            if len({t["artist_name"] for t in tracks}) < CFG["min_artists"]:
                continue
            ids = np.empty(len(tracks), dtype=np.int32)
            for j, t in enumerate(tracks):
                tid = t["track_uri"].rsplit(":", 1)[-1]
                m = meta.get(tid)
                if m is None:
                    meta[tid] = [t["track_name"], t["artist_name"], t.get("duration_ms", 0) or 0, 1]
                    id_of[tid] = len(id_of)
                else:
                    m[3] += 1
                ids[j] = id_of[tid]
            playlists.append(ids)
        if si % 200 == 0:
            print(f"  {si}/{len(slices)} | {len(playlists):,} playlists | {time.time()-t0:.0f}s", flush=True)
    tids = list(id_of.keys())
    pd.DataFrame({
        "track_id": tids,
        "name": [meta[t][0] for t in tids],
        "artist": [meta[t][1] for t in tids],
        "duration_ms": np.array([meta[t][2] for t in tids], dtype=np.int64),
        "playlist_count": np.array([meta[t][3] for t in tids], dtype=np.int32),
    }).to_parquet(meta_path, index=False)
    flat = np.concatenate(playlists)
    lens = np.array([len(p) for p in playlists], dtype=np.int32)
    np.savez(playlists_path, flat=flat, lens=lens)
    del meta, id_of, playlists

meta_df = pd.read_parquet(meta_path)
z = np.load(playlists_path)
flat, lens = z["flat"], z["lens"]
offsets = np.zeros(len(lens) + 1, dtype=np.int64)
np.cumsum(lens, out=offsets[1:])
def playlist_at(i):
    return flat[offsets[i]:offsets[i + 1]]
print(f"{len(lens):,} playlists | {len(meta_df):,} tracks")


## Stage 2 — embeddings, the INSTALLED 50-tree ANN artifact, mood, audio

In [ ]:
from gensim.models import KeyedVectors
from annoy import AnnoyIndex

wv = KeyedVectors.load(str(BASE / "item2vec.wordvectors"), mmap="r")
DIM = wv.vector_size
vecs = np.asarray(wv.vectors, dtype=np.float32)
norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms == 0] = 1.0
unit_vecs = vecs / norms

tid_arr = meta_df["track_id"].to_numpy()
i2v_row = np.array([wv.key_to_index.get(t, -1) for t in tid_arr], dtype=np.int64)
mpd_of_row = np.full(len(wv), -1, dtype=np.int64)
mapped = np.where(i2v_row >= 0)[0]
mpd_of_row[i2v_row[mapped]] = mapped

artist_norm = meta_df["artist"].fillna("").str.lower().str.strip().to_numpy()
pop = meta_df["playlist_count"].to_numpy()
log_pop_all = (np.log1p(pop) / np.log1p(pop.max())).astype(np.float32)
dur_all = meta_df["duration_ms"].fillna(0).to_numpy(np.float32)

# Serving parity: load the exact installed Annoy artifact (50 trees) and
# verify its id map matches item2vec key order. Never rebuild with 32 trees.
ann_hits = glob.glob("/kaggle/input/**/annoy.index", recursive=True)
assert ann_hits, "attach the one-rec-ann-index dataset (the installed 50-tree artifact)"
ann = AnnoyIndex(DIM, "angular")
ann.load(ann_hits[0])
id_map = json.load(open(glob.glob("/kaggle/input/**/id_map.json", recursive=True)[0]))
assert ann.get_n_items() == len(wv) == len(id_map)
assert id_map[:1000] == wv.index_to_key[:1000], "ANN id map does not match item2vec order"
print(f"installed ANN loaded: {ann.get_n_items():,} items")

mp = pickle.load(open(BASE / "mood_predictor.pkl", "rb"))
mood_model = mp["model"]
mood_order = [mp.get("feature_names", F.MOOD_DIMS).index(d) for d in F.MOOD_DIMS]
def predict_mood(rows):
    return np.clip(mood_model.predict(vecs[rows]), 0.0, 1.0)[:, mood_order].astype(np.float32)

audio_hits = glob.glob("/kaggle/input/**/audio_emb.parquet", recursive=True)
assert audio_hits, "attach the one-rec-audio-ckpt dataset"
adf = pd.read_parquet(audio_hits[0])
audio_mat = np.stack([np.frombuffer(b, dtype=np.float16).astype(np.float32) for b in adf["embedding"]])
a_norms = np.linalg.norm(audio_mat, axis=1, keepdims=True); a_norms[a_norms == 0] = 1.0
audio_mat = audio_mat / a_norms
audio_row = meta_df["track_id"].map(pd.Series(np.arange(len(adf)), index=adf["track_id"])).fillna(-1).astype(np.int64).to_numpy()
print(f"audio: {len(adf):,} tracks ({(audio_row >= 0).mean():.1%} of MPD meta)")

# Same-artist audio proxy, mirroring engine._seed_audio: most popular
# audio-covered track per artist.
_has_audio_mpd = np.where(audio_row >= 0)[0]
_order = _has_audio_mpd[np.argsort(-pop[_has_audio_mpd])]
artist_audio_proxy = {}
for m in _order[::-1]:   # reverse so the MOST popular ends up winning
    artist_audio_proxy[artist_norm[m]] = int(m)
print(f"artist audio proxies: {len(artist_audio_proxy):,} artists")


## Stage 3 — deterministic query manifest (before NCF, so nothing leaks)

Per-playlist RNG keyed by playlist id: seed/holdout/mode assignments are
reproducible regardless of execution order or cache state.


In [ ]:
rng_split = np.random.default_rng(CFG["seed"] + 5)
eligible = [i for i in range(len(lens)) if (i2v_row[playlist_at(i)] >= 0).sum() >= 12]
perm = rng_split.permutation(len(eligible))
need = CFG["ranker_train"] + CFG["ranker_val"] + CFG["ranker_test"]
assert len(eligible) >= need
splits = {
    "train": [eligible[j] for j in perm[:CFG["ranker_train"]]],
    "val":   [eligible[j] for j in perm[CFG["ranker_train"]:CFG["ranker_train"] + CFG["ranker_val"]]],
    "test":  [eligible[j] for j in perm[CFG["ranker_train"] + CFG["ranker_val"]:need]],
}
all_split_pids = set(splits["train"]) | set(splits["val"]) | set(splits["test"])

def pl_rng(pl_idx):
    return np.random.default_rng(CFG["seed"] * 1_000_003 + int(pl_idx))

def manifest_entry(pl_idx, mode):
    """Deterministic (seed, context, positives) for one query.
    assisted: context = 80% of in-vocab tracks, seed = one holdout track,
              positives = remaining holdout tracks.
    seed_only: context = EMPTY, positives = ALL other unique in-vocab tracks
               (no oracle context is hidden from the candidate set)."""
    tracks = playlist_at(pl_idx)
    tracks = tracks[i2v_row[tracks] >= 0]
    tracks = pd.unique(tracks)
    r = pl_rng(pl_idx)
    order = r.permutation(len(tracks))
    n_hold = max(2, int(0.2 * len(order)))
    seed_mpd = int(tracks[order[0]])
    if mode == "seed_only":
        context = np.array([], dtype=tracks.dtype)
        positives = np.array([t for t in tracks if t != seed_mpd], dtype=tracks.dtype)
    else:
        holdout = tracks[order[:n_hold]]
        context = tracks[order[n_hold:]]
        positives = holdout[1:]
    return seed_mpd, context, positives

manifest = []   # (pl_idx, split, mode)
for split_name, pids in splits.items():
    for n, pi in enumerate(pids):
        if split_name == "train":
            mode = "seed_only" if (n / len(pids)) < CFG["seed_only_frac"] else "assisted"
        elif split_name == "test":
            mode = "assisted" if n < len(pids) // 2 else "seed_only"
        else:
            mode = "assisted"
        manifest.append((int(pi), split_name, mode))
print(f"manifest: {len(manifest):,} queries | "
      f"{sum(1 for _, s, m in manifest if m == 'seed_only'):,} seed-only")


## Stage 4 — item-NCF, fully excluded from ALL ranker splits

In [ ]:
ncf_path = WORK / "ncf_item_v2.pt"
device = "cpu"
if torch.cuda.is_available():
    try:
        (torch.ones(2, device="cuda") * 2).sum().item()
        device = "cuda"
    except Exception as exc:
        print(f"CUDA unusable ({type(exc).__name__}) — CPU")
print("NCF device:", device)

# Vocabulary and popular-negative pool come from BACKGROUND playlists only.
background = [i for i in range(len(lens)) if i not in all_split_pids]
bg_counts = np.zeros(len(meta_df), dtype=np.int64)
for i in background:
    bg_counts[playlist_at(i)] += 1
vocab_mpd = np.where(bg_counts >= CFG["ncf_min_count"])[0]
ncf_index_of = np.full(len(meta_df), -1, dtype=np.int64)
ncf_index_of[vocab_mpd] = np.arange(len(vocab_mpd))
N_NCF = len(vocab_mpd)
print(f"NCF vocab (background-only counts): {N_NCF:,}")

NCF_CONFIG = {"n_tracks": N_NCF, "gmf_dim": CFG["ncf_gmf_dim"],
              "mlp_dim": CFG["ncf_mlp_dim"], "mlp_layers": CFG["ncf_mlp_layers"]}

class ItemNCF(torch.nn.Module):   # mirrors services/recommendation/ncf.py
    def __init__(self, config):
        super().__init__()
        nn = torch.nn
        n = config["n_tracks"]
        self.gmf_emb = nn.Embedding(n, config["gmf_dim"])
        self.mlp_emb = nn.Embedding(n, config["mlp_dim"])
        layers, in_dim = [], config["mlp_dim"] * 2
        for out_dim in config["mlp_layers"]:
            layers += [nn.Linear(in_dim, out_dim), nn.ReLU()]
            in_dim = out_dim
        self.mlp = nn.Sequential(*layers)
        self.head = nn.Linear(config["gmf_dim"] + in_dim, 1)

def score_batch(model, ctx_pad, ctx_mask, cand):
    m = ctx_mask.unsqueeze(-1)
    denom = ctx_mask.sum(1, keepdim=True).clamp(min=1.0)
    u_gmf = (model.gmf_emb(ctx_pad) * m).sum(1) / denom
    u_mlp = (model.mlp_emb(ctx_pad) * m).sum(1) / denom
    gmf_out = u_gmf * model.gmf_emb(cand)
    mlp_out = model.mlp(torch.cat([u_mlp, model.mlp_emb(cand)], dim=1))
    return model.head(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)


In [ ]:
if ncf_path.exists():
    print("stage 4 cached")
else:
    rng = np.random.default_rng(CFG["seed"])
    pl_vocab = []
    for i in background:
        v = ncf_index_of[playlist_at(i)]
        v = v[v >= 0]
        if len(v) >= CFG["ncf_ctx_min"] + 1:
            pl_vocab.append(v.astype(np.int64))
    print(f"{len(pl_vocab):,} background playlists for NCF")

    popular_pool = ncf_index_of[vocab_mpd[np.argsort(-bg_counts[vocab_mpd])[:max(N_NCF // 5, 1)]]]

    def gen_batch(B):
        L = CFG["ncf_ctx_max"]
        ctx_pad = np.zeros((B, L), dtype=np.int64)
        ctx_mask = np.zeros((B, L), dtype=np.float32)
        pos = np.empty(B, dtype=np.int64)
        neg = np.empty(B, dtype=np.int64)
        pls = rng.integers(0, len(pl_vocab), B)
        hard = rng.random(B) < 0.5
        neg[hard] = rng.choice(popular_pool, hard.sum())
        neg[~hard] = rng.integers(0, N_NCF, (~hard).sum())
        for b, pi in enumerate(pls):
            tracks = pl_vocab[pi]
            k = int(rng.integers(CFG["ncf_ctx_min"], min(CFG["ncf_ctx_max"], len(tracks) - 1) + 1))
            picks = rng.choice(len(tracks), k + 1, replace=False)
            pos[b] = tracks[picks[0]]
            ctx = tracks[picks[1:]]
            ctx_pad[b, :len(ctx)] = ctx
            ctx_mask[b, :len(ctx)] = 1.0
            # Reroll false negatives WITHIN the original branch (audit: uniform
            # rerolls eroded the popular/hard-negative mixture).
            tracks_set = set(tracks.tolist())
            while int(neg[b]) in tracks_set:
                neg[b] = rng.choice(popular_pool) if hard[b] else rng.integers(0, N_NCF)
        return (torch.from_numpy(ctx_pad).to(device), torch.from_numpy(ctx_mask).to(device),
                torch.from_numpy(pos).to(device), torch.from_numpy(neg).to(device))

    model = ItemNCF(NCF_CONFIG).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=CFG["ncf_lr"])
    steps = CFG["ncf_samples_per_epoch"] // CFG["ncf_batch"]
    for epoch in range(CFG["ncf_epochs"]):
        model.train(); running = 0.0; t0 = time.time()
        for step in range(steps):
            ctx_pad, ctx_mask, pos_t, neg_t = gen_batch(CFG["ncf_batch"])
            loss = -torch.nn.functional.logsigmoid(
                score_batch(model, ctx_pad, ctx_mask, pos_t)
                - score_batch(model, ctx_pad, ctx_mask, neg_t)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            running += loss.item()
            if (step + 1) % 200 == 0:
                print(f"  e{epoch+1} {step+1}/{steps} loss {running/200:.4f} ({time.time()-t0:.0f}s)", flush=True)
                running = 0.0
    model.eval().cpu()
    torch.save({"format": "item-ncf-v2", "state_dict": model.state_dict(),
                "track_id_map": {tid_arr[m]: int(i) for i, m in enumerate(vocab_mpd)},
                "config": NCF_CONFIG,
                "provenance": {"background_only": True, "n_background": len(background),
                                "split_hash": hashlib.sha256(json.dumps(
                                    {k: sorted(v) for k, v in splits.items()}).encode()).hexdigest()}},
               ncf_path)
    print("saved", ncf_path)

ckpt = torch.load(ncf_path, map_location="cpu", weights_only=False)
ncf_model = ItemNCF(ckpt["config"]); ncf_model.load_state_dict(ckpt["state_dict"]); ncf_model.eval()

@torch.inference_mode()
def ncf_score(context_mpd, cand_mpd):
    ctx = ncf_index_of[context_mpd]; ctx = ctx[ctx >= 0]
    if len(ctx) < 3:
        return None, None
    cand = ncf_index_of[cand_mpd]
    ok = cand >= 0
    scores = np.zeros(len(cand_mpd), dtype=np.float32)
    if ok.any():
        ctx_pad = torch.from_numpy(ctx).unsqueeze(0)
        u_m = torch.ones(1, len(ctx)).unsqueeze(-1)
        u_gmf = (ncf_model.gmf_emb(ctx_pad) * u_m).sum(1) / len(ctx)
        u_mlp = (ncf_model.mlp_emb(ctx_pad) * u_m).sum(1) / len(ctx)
        cand_t = torch.from_numpy(cand[ok])
        gmf_out = u_gmf * ncf_model.gmf_emb(cand_t)
        mlp_out = ncf_model.mlp(torch.cat([u_mlp.expand(len(cand_t), -1), ncf_model.mlp_emb(cand_t)], dim=1))
        s = ncf_model.head(torch.cat([gmf_out, mlp_out], dim=1)).squeeze(1)
        scores[ok] = torch.sigmoid(s).numpy()
    return scores, ok.astype(np.float32)


## Stage 5 — group construction: serving-exact, 5-stage funnel, no oracle context


In [ ]:
K_SEED = int(CFG["retrieval_pool"] * CFG["seed_share"])
K_PLAYLIST = CFG["retrieval_pool"] - K_SEED


def build_group(pl_idx, mode):
    """Returns (payload_or_None, funnel). payload = dict with X, y (binary),
    y_graded (diagnostic), artists, n_orig_pos."""
    seed_mpd, context, positives_arr = manifest_entry(pl_idx, mode)
    positives = set(int(p) for p in positives_arr)
    funnel = {"positives": len(positives)}
    seed_row = int(i2v_row[seed_mpd])

    # --- retrieval (serving-exact: seed-only gets the whole pool) ---
    k_seed = CFG["retrieval_pool"] if mode == "seed_only" else K_SEED
    seed_nb = [r for r in ann.get_nns_by_item(seed_row, k_seed + 1) if r != seed_row][:k_seed]
    ctx_rows = i2v_row[context]
    if mode == "seed_only":
        pl_nb, mean_vec = [], None
    else:
        mean_vec = vecs[ctx_rows].mean(axis=0)
        pl_nb = ann.get_nns_by_vector(mean_vec, K_PLAYLIST)

    seed_rank = {r: k for k, r in enumerate(seed_nb)}
    pl_rank = {r: k for k, r in enumerate(pl_nb)}
    exclude = set(ctx_rows.tolist()); exclude.add(seed_row)
    cand_rows, seen = [], set()
    for r in [*seed_nb, *pl_nb]:
        if r not in exclude and r not in seen:
            seen.add(r); cand_rows.append(r)
    if not cand_rows:
        funnel.update(retrieved=0, after_artist=0, after_mood=0, after_cap=0)
        return None, funnel
    cand_rows = np.array(cand_rows, dtype=np.int64)
    cand_mpd = mpd_of_row[cand_rows]
    pos_list = list(positives)
    funnel["retrieved"] = int(np.isin(cand_mpd, pos_list).sum())

    cand_artists = np.where(cand_mpd >= 0, artist_norm[np.maximum(cand_mpd, 0)], "")
    if CFG["artist_precap"] is not None:
        keep, counts_a = [], {}
        for j, a in enumerate(cand_artists):
            if a:
                if counts_a.get(a, 0) >= CFG["artist_precap"]:
                    continue
                counts_a[a] = counts_a.get(a, 0) + 1
            keep.append(j)
        cand_rows, cand_mpd, cand_artists = cand_rows[keep], cand_mpd[keep], cand_artists[keep]
    funnel["after_artist"] = int(np.isin(cand_mpd, pos_list).sum())

    moods = predict_mood(cand_rows)
    seed_mood = predict_mood(np.array([seed_row]))[0]
    if mode == "seed_only" or not len(ctx_rows):
        target_mood = seed_mood
    else:
        target_mood = 0.8 * seed_mood + 0.2 * predict_mood(ctx_rows[:50]).mean(axis=0)
    if CFG["mood_keep"] < 1.0:
        sim = 1.0 - np.abs(moods - target_mood).mean(axis=1)
        keep_n = int(len(cand_rows) * CFG["mood_keep"])
        keep = np.sort(np.argsort(-sim)[:keep_n])
        cand_rows, cand_mpd, cand_artists, moods = cand_rows[keep], cand_mpd[keep], cand_artists[keep], moods[keep]
    funnel["after_mood"] = int(np.isin(cand_mpd, pos_list).sum())

    if len(cand_rows) > CFG["cand_cap"]:
        cand_rows, cand_mpd = cand_rows[:CFG["cand_cap"]], cand_mpd[:CFG["cand_cap"]]
        cand_artists, moods = cand_artists[:CFG["cand_cap"]], moods[:CFG["cand_cap"]]
    funnel["after_cap"] = int(np.isin(cand_mpd, pos_list).sum())

    is_pos = np.isin(cand_mpd, pos_list)
    if not is_pos.any():
        return None, funnel

    y = is_pos.astype(np.float32)
    seed_sim = unit_vecs[cand_rows] @ unit_vecs[seed_row]
    y_graded = np.where(is_pos & (seed_sim >= 0.75), 3.0,
               np.where(is_pos & (seed_sim >= 0.55), 2.0,
               np.where(is_pos, 1.0, 0.0))).astype(np.float32)

    known = cand_mpd >= 0
    ncf_ctx = np.array([seed_mpd]) if mode == "seed_only" else np.concatenate([context, [seed_mpd]])
    ncf_s, ncf_m = ncf_score(ncf_ctx, np.maximum(cand_mpd, 0))
    if ncf_s is not None:
        ncf_s, ncf_m = np.where(known, ncf_s, 0.0), np.where(known, ncf_m, 0.0)

    a_rows = np.where(cand_mpd >= 0, audio_row[np.maximum(cand_mpd, 0)], -1)
    audio_vecs = np.zeros((len(cand_rows), audio_mat.shape[1]), dtype=np.float32)
    got = a_rows >= 0
    audio_vecs[got] = audio_mat[a_rows[got]]
    seed_a = audio_row[seed_mpd]
    if seed_a >= 0:
        seed_audio_vec = audio_mat[seed_a]
    else:  # serving-exact same-artist proxy
        proxy_m = artist_audio_proxy.get(str(artist_norm[seed_mpd]))
        seed_audio_vec = audio_mat[audio_row[proxy_m]] if proxy_m is not None else None
    playlist_audio_mean = None
    if mode != "seed_only":
        ctx_a = audio_row[context[:100]]; ctx_a = ctx_a[ctx_a >= 0]   # serving: first 100
        playlist_audio_mean = audio_mat[ctx_a].mean(axis=0) if len(ctx_a) else None

    if mode == "seed_only":
        shares, mean_dur, playlist_vecs = {}, 0.0, None
    else:
        s = pd.Series(artist_norm[context]).value_counts() / len(context)
        shares = {a: float(v) for a, v in s.items() if a}
        d = dur_all[context]; d = d[d > 0]
        mean_dur = float(d.mean()) if len(d) else 0.0
        playlist_vecs = vecs[ctx_rows[:100]]

    ctx_obj = F.RankingContext(
        seed_vec=vecs[seed_row], playlist_mean_vec=mean_vec, playlist_vecs=playlist_vecs,
        seed_rank={int(r): k for r, k in seed_rank.items()},
        playlist_rank={int(r): k for r, k in pl_rank.items()},
        seed_artist=str(artist_norm[seed_mpd]), playlist_artist_share=shares,
        target_mood=target_mood, playlist_mean_duration_ms=mean_dur,
        seed_audio_vec=seed_audio_vec, playlist_audio_mean=playlist_audio_mean,
    )
    X = F.build_matrix(
        ids=[int(r) for r in cand_rows], vectors=vecs[cand_rows], artists=list(cand_artists),
        log_pop=np.where(known, log_pop_all[np.maximum(cand_mpd, 0)], 0.0).astype(np.float32),
        durations_ms=np.where(known, dur_all[np.maximum(cand_mpd, 0)], 0.0).astype(np.float32),
        moods=moods, ncf_scores=ncf_s, ncf_mask=ncf_m, ctx=ctx_obj, audio_vecs=audio_vecs,
    )
    return {"X": X.to_numpy(np.float32), "y": y, "y_graded": y_graded,
            "artists": cand_artists, "n_orig_pos": len(positives)}, funnel


In [ ]:
def config_hash():
    payload = {"features": F.FEATURE_NAMES,
               "config": {k: (v if not isinstance(v, list) else list(v)) for k, v in sorted(CFG.items())},
               "split_hash": hashlib.sha256(json.dumps({k: sorted(v) for k, v in splits.items()}).encode()).hexdigest()}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

RUN_HASH = config_hash()
print("run hash:", RUN_HASH[:16])

def build_split(name):
    cache = WORK / f"groups_{name}_{RUN_HASH[:12]}.pkl"   # cache key includes config hash
    if cache.exists():
        with open(cache, "rb") as f:
            return pickle.load(f)
    queries = [(pi, m) for pi, s, m in manifest if s == name]
    groups, dropped, funnel_tot, t0 = [], [],         {"positives": 0, "retrieved": 0, "after_artist": 0, "after_mood": 0, "after_cap": 0}, time.time()
    for n, (pi, mode) in enumerate(queries):
        payload, funnel = build_group(pi, mode)
        for k in funnel_tot:
            funnel_tot[k] += funnel.get(k, 0)
        if payload is None:
            dropped.append({"pl_idx": pi, "mode": mode, "n_orig_pos": funnel["positives"]})
        else:
            payload["mode"] = mode
            payload["pl_idx"] = pi
            groups.append(payload)
        if (n + 1) % 2000 == 0:
            print(f"  {name}: {n+1}/{len(queries)} ({time.time()-t0:.0f}s, {len(dropped)} dropped)", flush=True)
    out = {"groups": groups, "dropped": dropped, "funnel": funnel_tot}
    with open(cache, "wb") as f:
        pickle.dump(out, f)
    p = funnel_tot["positives"] or 1
    print(f"{name}: {len(groups):,} kept, {len(dropped):,} dropped | funnel "
          f"{funnel_tot['retrieved']/p:.1%} -> {funnel_tot['after_artist']/p:.1%} -> "
          f"{funnel_tot['after_mood']/p:.1%} -> {funnel_tot['after_cap']/p:.1%}")
    return out

train_data = build_split("train")
val_data = build_split("val")
test_data = build_split("test")


## Stage 6 — ranker variants (binary primary + ablations + graded diagnostic)

In [ ]:
def stack(groups, label_key="y", zero_cols=()):
    X = np.concatenate([g["X"] for g in groups])
    if zero_cols:
        X = X.copy()
        for c in zero_cols:
            X[:, F.FEATURE_NAMES.index(c)] = 0.0
    y = np.concatenate([g[label_key] for g in groups])
    grp = np.array([len(g["y"]) for g in groups], dtype=np.int32)
    return X, y, grp

AUDIO_COLS = ("audio_cos_seed", "audio_cos_playlist", "has_audio")
MOOD_COLS = ("valence_diff", "energy_diff", "acousticness_diff", "danceability_diff", "mood_sim")

params = dict(objective="lambdarank", metric="ndcg", ndcg_eval_at=[10, 50],
              learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
              feature_fraction=0.9, lambdarank_truncation_level=50,
              verbosity=-1, seed=CFG["seed"])

def train_variant(name, label_key="y", zero_cols=()):
    Xtr, ytr, gtr = stack(train_data["groups"], label_key, zero_cols)
    Xv, yv, gv = stack(val_data["groups"], label_key, zero_cols)
    ds = lgb.Dataset(Xtr, label=ytr.astype(np.int32), group=gtr, feature_name=F.FEATURE_NAMES)
    vds = lgb.Dataset(Xv, label=yv.astype(np.int32), group=gv, feature_name=F.FEATURE_NAMES, reference=ds)
    booster = lgb.train(params, ds, num_boost_round=500, valid_sets=[vds],
                        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    print(f"{name}: {booster.num_trees()} trees")
    return booster

variants = {
    "binary_full": train_variant("binary_full"),
    "binary_no_audio": train_variant("binary_no_audio", zero_cols=AUDIO_COLS),
    "binary_no_mood": train_variant("binary_no_mood", zero_cols=MOOD_COLS),
    "graded_diagnostic": train_variant("graded_diagnostic", label_key="y_graded"),
}
variants["binary_full"].save_model(str(WORK / "lightgbm_ranker_v2.txt"))
assert lgb.Booster(model_file=str(WORK / "lightgbm_ranker_v2.txt")).feature_name() == F.FEATURE_NAMES

# v9 baseline, rescored on this exact corrected manifest
v9_hits = glob.glob("/kaggle/input/**/v9_baseline_ranker.txt", recursive=True)
v9_baseline = lgb.Booster(model_file=v9_hits[0]) if v9_hits else None
print("v9 baseline:", "loaded" if v9_baseline else "NOT ATTACHED")


## Stage 7 — true end-to-end evaluation + validation-only policy grid

Every metric uses the ORIGINAL positive count as denominator and includes
dropped groups as zeros. The policy simulates exact serving: standardization,
dials, eligible-first floor, overfetch, and the service's one-per-artist rule.


In [ ]:
def policy_order(scores, seed_cos, log_pop, lam, mu, floor):
    s = np.asarray(scores, dtype=np.float64)
    if lam or mu:
        s = (s - s.mean()) / (s.std() or 1.0)
    s = s + lam * seed_cos - mu * log_pop
    order = np.argsort(-s)
    if floor:
        elig = order[seed_cos[order] >= floor]
        inelig = order[seed_cos[order] < floor]
        order = np.concatenate([elig, inelig])
    return order

def served_topk(order, artists, k, overfetch):
    """Service simulation: overfetch then one-per-artist (unknown artists pass)."""
    pool = order[:k * overfetch]
    out, seen = [], set()
    for i in pool:
        a = artists[i]
        if a:
            if a in seen:
                continue
            seen.add(a)
        out.append(i)
        if len(out) == k:
            break
    return np.array(out, dtype=np.int64)

def true_group_metrics(top_idx, y, n_orig_pos, seed_cos):
    rel = y[top_idx]
    k = len(top_idx)
    hits = float(rel.sum())
    dcg = float((rel / np.log2(np.arange(2, k + 2))).sum())
    n_ideal = min(n_orig_pos, k)
    idcg = float((np.ones(n_ideal) / np.log2(np.arange(2, n_ideal + 2))).sum())
    pos = np.nonzero(rel > 0)[0]
    return {"recall@10": hits / max(n_orig_pos, 1),
            "ndcg@10": dcg / idcg if idcg > 0 else 0.0,
            "hit@1": float(rel[0] > 0) if k else 0.0,
            "mrr": float(1.0 / (pos[0] + 1)) if len(pos) else 0.0,
            "hits": hits,
            "seed_cos@10": float(seed_cos[top_idx].mean()) if k else 0.0}

SEED_COS_COL = F.FEATURE_NAMES.index("seed_i2v_cos")
LOG_POP_COL = F.FEATURE_NAMES.index("log_pop")

def evaluate_policy(booster, data, lam, mu, floor, k=10):
    """True end-to-end: dropped groups contribute zeros with their orig positives."""
    per_group, total_hits, total_pos = [], 0.0, 0
    for g in data["groups"]:
        X, y = g["X"], g["y"]
        seed_cos, log_pop = X[:, SEED_COS_COL], X[:, LOG_POP_COL]
        order = policy_order(booster.predict(X), seed_cos, log_pop, lam, mu, floor)
        top = served_topk(order, g["artists"], k, CFG["serve_overfetch"])
        m = true_group_metrics(top, y, g["n_orig_pos"], seed_cos)
        per_group.append(m)
        total_hits += m["hits"]; total_pos += g["n_orig_pos"]
    for d in data["dropped"]:
        per_group.append({"recall@10": 0.0, "ndcg@10": 0.0, "hit@1": 0.0, "mrr": 0.0,
                          "hits": 0.0, "seed_cos@10": np.nan})
        total_pos += d["n_orig_pos"]
    macro = {key: float(np.mean([m[key] for m in per_group])) for key in ("recall@10", "ndcg@10", "hit@1", "mrr")}
    macro["micro_recall@10"] = total_hits / max(total_pos, 1)
    macro["seed_cos@10"] = float(np.nanmean([m["seed_cos@10"] for m in per_group]))
    macro["coverage"] = len(data["groups"]) / (len(data["groups"]) + len(data["dropped"]))
    return macro

# ---- validation-only grid; freeze best policy before touching test ----
grid_results = []
for lam in CFG["grid_lambda"]:
    for mu in CFG["grid_mu"]:
        for floor in CFG["grid_floor"]:
            m = evaluate_policy(variants["binary_full"], val_data, lam, mu, floor)
            grid_results.append({"lam": lam, "mu": mu, "floor": floor, **m})
grid_df = pd.DataFrame(grid_results)
ok = grid_df[grid_df["seed_cos@10"] >= CFG["policy_seed_cos_min"]]
chosen = (ok if len(ok) else grid_df).sort_values("ndcg@10", ascending=False).iloc[0]
POLICY = {"lam": float(chosen["lam"]), "mu": float(chosen["mu"]), "floor": float(chosen["floor"])}
print("frozen policy:", POLICY, f"| val ndcg@10 {chosen['ndcg@10']:.4f} seed_cos {chosen['seed_cos@10']:.4f}")
print(grid_df.sort_values("ndcg@10", ascending=False).head(8).round(4).to_string())


In [ ]:
# ---- test scoring: variants + v9 baseline, all through the frozen policy ----
def eval_by_mode(booster, data, policy):
    out = {}
    for mode in ("assisted", "seed_only"):
        sub = {"groups": [g for g in data["groups"] if g["mode"] == mode],
               "dropped": [d for d in data["dropped"] if d["mode"] == mode]}
        out[mode] = evaluate_policy(booster, sub, policy["lam"], policy["mu"], policy["floor"])
    return out

report = {}
for name, booster in variants.items():
    report[name] = eval_by_mode(booster, test_data, POLICY)
if v9_baseline is not None:
    report["v9_baseline"] = eval_by_mode(v9_baseline, test_data, POLICY)
report["raw_binary_full_nopolicy"] = eval_by_mode(variants["binary_full"], test_data,
                                                  {"lam": 0.0, "mu": 0.0, "floor": 0.0})

for name, modes in report.items():
    a, s = modes["assisted"], modes["seed_only"]
    print(f"{name:26s} assisted ndcg {a['ndcg@10']:.4f} recall {a['recall@10']:.4f} hit1 {a['hit@1']:.4f} | "
          f"seed-only ndcg {s['ndcg@10']:.4f} hit1 {s['hit@1']:.4f}")

importance = dict(zip(F.FEATURE_NAMES, variants["binary_full"].feature_importance("gain").round(1).tolist()))


## Stage 8 — package artifacts + provenance

In [ ]:
import shutil
shutil.copy(audio_hits[0], WORK / "audio_emb.parquet")

# eval sample: assisted test groups with binary labels + artists for local A/B
rows = []
for qid, g in enumerate([g for g in test_data["groups"] if g["mode"] == "assisted"][:500]):
    df = pd.DataFrame(g["X"], columns=F.FEATURE_NAMES)
    df["label"] = g["y"]
    df["qid"] = qid
    df["artist"] = g["artists"]
    df["n_orig_pos"] = g["n_orig_pos"]
    rows.append(df)
pd.concat(rows).to_parquet(WORK / "eval_sample.parquet", index=False)

group_manifest = pd.DataFrame(
    [{"pl_idx": g["pl_idx"], "mode": g["mode"], "n_orig_pos": g["n_orig_pos"], "kept": True}
     for g in test_data["groups"]] +
    [{"pl_idx": d["pl_idx"], "mode": d["mode"], "n_orig_pos": d["n_orig_pos"], "kept": False}
     for d in test_data["dropped"]])
group_manifest.to_parquet(WORK / "group_manifest.parquet", index=False)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

json.dump({
    "metrics": report["binary_full"]["assisted"],          # headline: true e2e assisted
    "metrics_by_variant": report,
    "frozen_policy": POLICY,
    "policy_grid_val": grid_df.round(5).to_dict("records"),
    "funnel": {"train": train_data["funnel"], "val": val_data["funnel"], "test": test_data["funnel"]},
    "feature_importance_gain": importance,
    "config": {k: v for k, v in CFG.items() if isinstance(v, (int, float, str, bool))},
    "dataset": {"train_groups": len(train_data["groups"]), "val_groups": len(val_data["groups"]),
                "test_groups": len(test_data["groups"]), "test_dropped": len(test_data["dropped"]),
                "ncf_vocab": int(N_NCF)},
    "run_hash": RUN_HASH,
    "artifact_sha256": {p.name: sha256_file(p) for p in
                        [WORK / "lightgbm_ranker_v2.txt", WORK / "ncf_item_v2.pt",
                         WORK / "track_meta.parquet", WORK / "audio_emb.parquet"]},
}, open(WORK / "metrics.json", "w"), indent=2)

json.dump(F.FEATURE_NAMES, open(WORK / "feature_names.json", "w"))
with zipfile.ZipFile(WORK / "artifacts.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["lightgbm_ranker_v2.txt", "ncf_item_v2.pt", "track_meta.parquet",
              "metrics.json", "feature_names.json", "eval_sample.parquet",
              "audio_emb.parquet", "group_manifest.parquet"]:
        zf.write(WORK / f, f)
print("artifacts.zip:", f"{(WORK / 'artifacts.zip').stat().st_size / 1e6:.1f} MB | run", RUN_HASH[:16])
